# 🎵 Proyecto ML: Predicción de Popularidad en Spotify

## Fase 1 — Dataset, Objetivos y Cronograma

**Curso:** Aprendizaje Automático  
**Dataset:** [Spotify Tracks Dataset — Kaggle (Maharshi Pandya)](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)  
**Integrantes:** [nombre(s)]

---

## 1. Descripción del Dataset

El dataset contiene información de más de **114,000 tracks** de Spotify distribuidos en **125 géneros musicales**. Cada pista incluye métricas de audio calculadas por la API de Spotify, así como metadatos del artista y álbum.

### Variables principales

| Variable | Tipo | Descripción |
|---|---|---|
| `popularity` | int (0–100) | **Target**: popularidad de la canción en Spotify |
| `danceability` | float (0–1) | Qué tan bailable es la pista |
| `energy` | float (0–1) | Intensidad y actividad percibida |
| `loudness` | float (dB) | Volumen promedio en decibeles |
| `speechiness` | float (0–1) | Presencia de palabras habladas |
| `acousticness` | float (0–1) | Confianza en que la pista es acústica |
| `instrumentalness` | float (0–1) | Predicción de ausencia de voz |
| `liveness` | float (0–1) | Probabilidad de que sea en vivo |
| `valence` | float (0–1) | Positividad musical transmitida |
| `tempo` | float (BPM) | Velocidad estimada en beats por minuto |
| `duration_ms` | int | Duración en milisegundos |
| `track_genre` | str | Género musical |
| `explicit` | bool | Si contiene lenguaje explícito |

## 2. Objetivos del Proyecto

### Objetivo principal
> **Predecir la popularidad de una canción en Spotify a partir de sus características de audio.**

### Objetivos específicos

1. **Análisis exploratorio:** identificar qué features de audio están más correlacionadas con la popularidad; detectar outliers y distribuciones atípicas.
2. **Preprocesamiento:** normalizar variables, manejar valores nulos, codificar variables categóricas (`track_genre`, `explicit`).
3. **Modelado — Técnica 1:** Regresión Lineal / Polinomial para predecir `popularity` de forma continua.
4. **Modelado — Técnica 2:** Regresión Ridge o Lasso para controlar la multicolinealidad esperada entre features de audio (e.g., `energy` vs `loudness`).
5. **Comparación:** evaluar ambas técnicas con métricas MSE, R² y validación cruzada.

### Variable objetivo (target)
- **`popularity`** (regresión continua, rango 0–100)
- Alternativa clasificación: binarizar en `popular` (≥ 60) vs `no popular` (< 60)

## 3. Cronograma — 8 Semanas

| Semana | Entregable | Contenido |
|---|---|---|
| 1–2 | **Avance #1** ✅ | Dataset, objetivos, cronograma |
| 3–4 | **Avance #2** | EDA completo, preprocesamiento, selección de variables, entrenamiento técnica base |
| 5–6 | Desarrollo | Entrenamiento técnica 2, tuning de hiperparámetros |
| 7 | Pre-entrega | Comparación de modelos, análisis de errores |
| 8 | **Entrega final** | Reporte, notebook limpio, comparación ≥ 2 técnicas |

## 4. Instalación y Descarga del Dataset

Usamos `kagglehub`, la forma más simple de descargar datasets de Kaggle.

> **Requisito:** tener una cuenta en [kaggle.com](https://www.kaggle.com). La primera vez que corras esta celda, Colab te pedirá tu **usuario** y **API key** de Kaggle.
>
> Para obtener tu API key: kaggle.com → Settings → API → **Create New Token** → copia el `username` y `key` del archivo `kaggle.json` que se descarga.

In [ ]:
# Instalar kagglehub
!pip install kagglehub -q
print('✅ kagglehub instalado')

In [ ]:
import kagglehub
import os

# Descargar dataset (la primera vez pedirá usuario y API key)
path = kagglehub.dataset_download('maharshipandya/-spotify-tracks-dataset')
print('Path to dataset files:', path)

# Encontrar el CSV dentro de la carpeta descargada
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print('Archivos CSV encontrados:', csv_files)

DATASET_PATH = os.path.join(path, csv_files[0])
print('✅ Dataset listo en:', DATASET_PATH)

## 5. Carga e Inspección Inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('✅ Librerías importadas')

In [ ]:
# Cargar el dataset desde la ruta descargada por kagglehub
df = pd.read_csv(DATASET_PATH, index_col=0)

print(f'Shape: {df.shape}')
print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
df.head()

In [ ]:
# Tipos de datos por columna
df.dtypes

## 6. Estadísticas Descriptivas (`.describe()`)

In [ ]:
# Variables numéricas
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

In [ ]:
# Variables categóricas
df.describe(include='object')

## 7. Valores Nulos

In [ ]:
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(2)

null_report = pd.DataFrame({'Nulos': nulls, 'Porcentaje (%)': nulls_pct})
null_report = null_report[null_report['Nulos'] > 0]

if null_report.empty:
    print('✅ No hay valores nulos en el dataset')
else:
    display(null_report)

## 8. Distribución del Target (`popularity`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histograma
axes[0].hist(df['popularity'], bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribución de Popularity', fontsize=13)
axes[0].set_xlabel('Popularity (0–100)')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(df['popularity'].mean(), color='crimson', linestyle='--',
                label=f'Media = {df["popularity"].mean():.1f}')
axes[0].axvline(df['popularity'].median(), color='orange', linestyle='--',
                label=f'Mediana = {df["popularity"].median():.1f}')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['popularity'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Boxplot de Popularity', fontsize=13)
axes[1].set_ylabel('Popularity (0–100)')
axes[1].set_xticks([])

plt.suptitle('Variable Objetivo: Popularity', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nEstadísticas de popularity:")
print(df['popularity'].describe())

## 9. Distribución de Features de Audio

In [ ]:
audio_features = ['danceability', 'energy', 'loudness', 'speechiness',
                  'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(audio_features):
    axes[i].hist(df[feat], bins=40, color='steelblue', edgecolor='white', linewidth=0.3, alpha=0.85)
    axes[i].set_title(feat, fontsize=11)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frecuencia')

plt.suptitle('Distribución de Features de Audio', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Matriz de Correlación

In [ ]:
num_cols = audio_features + ['duration_ms', 'popularity']
corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, center=0, ax=ax,
    annot_kws={'size': 8}, linewidths=0.5
)
ax.set_title('Matriz de Correlación — Features de Audio', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlaciones con el target
print('\n📊 Correlación de cada feature con popularity:')
corr_target = corr_matrix['popularity'].drop('popularity').sort_values(key=abs, ascending=False)
print(corr_target.to_string())

## 11. Top Géneros más Populares

In [ ]:
top_genres = df.groupby('track_genre')['popularity'].mean().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top_genres.index[::-1], top_genres.values[::-1], color='steelblue', alpha=0.85)
ax.set_xlabel('Popularidad Media')
ax.set_title('Top 20 Géneros por Popularidad Media', fontsize=13, fontweight='bold')

for bar, val in zip(bars, top_genres.values[::-1]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## 12. Observaciones Preliminares

> Completa esta sección con tus observaciones tras correr el notebook.

**Sobre el target (`popularity`):**
- Distribución sesgada hacia valores bajos, con muchas canciones con popularidad ≈ 0
- La media es aproximadamente XX y la mediana XX → sesgo [positivo/negativo]

**Sobre las features:**
- Las variables más correlacionadas con `popularity` son: [rellenar tras correr]
- Se observa posible multicolinealidad entre `energy` y `loudness` (correlación esperada > 0.7)
- Variables como `instrumentalness` y `speechiness` tienen distribuciones muy sesgadas → posible transformación necesaria

**Decisión de modelado:**
- **Técnica 1:** Regresión Lineal / Polinomial como baseline
- **Técnica 2:** Ridge o Lasso para manejar multicolinealidad y features irrelevantes

---

## ✅ Repositorio del Proyecto

- **Repositorio GitHub:** [enlace aquí]
- **Dataset fuente:** https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset

---
*Fase 1 completada — próximo paso: Avance #2 (Preprocesamiento y entrenamiento técnica base)*